# 04 · Native AHR: theory-based architecture and verified controller

**Purpose:** implement and validate the AHR mechanisms in *AHR Unified Mathematical Theory* before training a Go player. This notebook combines a continuous learned Queen/Worker/Scout core with an explicit finite controller whose transitions, hard constraints, and acceptance checks are inspectable.

**Inputs:** `AHR_Unified_Theory (1).pdf`, notebook 00's tagged recovery definitions, the project `.venv` / **AHR Go (.venv)** kernel, and small synthetic fixtures defined here. No dataset, holdout, pretrained model, or external playing engine is used.

**Outputs:** importable `ahr-export` cells; `config/04_architecture.json`; `reports/04_architecture_validation.json`; untrained snapshots in `checkpoints/architecture-04-theory-<source-hash>/`. Expected runtime: **under 5 minutes** on this instance. Every backward pass is diagnostic; there are **zero optimizer updates**.

**Run All:** select **AHR Go (.venv)** and **Restart Kernel and Run All**, from this workspace or `notebooks/`. Terminal execution is also documented below. A new kernel imports only tagged definitions from notebook 00, never its training smoke workflow. Exact interruption/resume instructions appear beside recovery validation.

**Scope:** a theory-based architecture implementation and bounded synthetic integration. This is not yet the Go encoder/player of notebook 06. Timing limits here are cooperative checks between operations, not a hard real-time guarantee. Model weights are random. No learned improvement or Go strength is established.

**Saved Run All result: PASS.** All 14 code cells executed in a fresh project kernel. The 50,820-parameter model passed CPU and both A100 checks, with all 47 parameter tensors receiving finite nonzero diagnostic gradients. The separate fresh-kernel restart reproduced mid-Scout continuation exactly. All six compatibility clauses and 16 recovery fault cases passed. **Zero optimizer updates; no training.**

## Theory-to-code map

Page numbers below are **physical PDF pages**, followed by section/equation references.

| Source | Implementation | Interpretation |
|---|---|---|
| pp. 25–30, §§4–5 | Typed finite Queen/Worker/Scout/task/action records; six QWS clauses and operational transitions | Exact checks for the declared small Boolean domain. Neural vectors supplement these records; vectors alone are not the formal roles. |
| pp. 26–27, Eqs. 4.1–4.4 | `N` Queen cycles, `K` Worker steps per Queen, `T` Scout steps per Worker; explicit `ResetW`, `ResetU`, summaries and memory carry | GRU updates and mean summaries are explicit implementation choices permitted by the typed update functions. |
| pp. 31–35, §6 | `clamp((1−evaporation)*pheromone + deposit)` using the **pre-update** Q/W/U states; pheromone log prior | Continuous memory, not the optional quantized finite memory. No finite-state theorem is claimed for this neural core. |
| pp. 36–40, §§7–8 | Finite learned `E_Q`, `E_W`, `E_U`; separate 0/+∞ feasibility; verifier-gated bounded search | Budgeted search returns a verified candidate or `UNKNOWN`. It does not claim the global minimum over unvisited trajectories. |
| pp. 55–57, §13.3 | Full autograd through short nested recurrence; explicit detach only for stored state | No constant-memory or unbiased detached-gradient claim. |
| pp. 80–85, §§20–23 | No fitting; hard acceptance remains outside learned scores | Ranking quality, safe learned pruning, coverage, calibrated halting and Go transfer remain unmeasured. |

Stigmergy is optional in the unified definition (p. 85); this implementation includes its canonical continuous form. Adaptive learned halting, learned admissible pruning bounds, low-rank memory and quantization are not required for this selected instantiation.

The neural path proposes/ranks; the finite controller authorizes, constructs and verifies. Passing a finite constraint fixture means that particular predicate was checked. For Go, a legal move certificate will establish **legality**, not optimal play or a proved win.

## 1. Workspace and definition-only recovery imports

In [ ]:
from pathlib import Path
import os, json, hashlib, time, platform, subprocess, copy
import nbformat

PROJECT = Path(os.environ.get("AHR_GO_PROJECT", Path.cwd())).resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
NOTEBOOK_PATH = PROJECT / "notebooks/04_native_ahr_architecture.ipynb"
THEORY_PATH = PROJECT / "AHR_Unified_Theory (1).pdf"
if not THEORY_PATH.is_file():
    raise RuntimeError("The supplied AHR theory PDF is required for source provenance")
upstream = nbformat.read(PROJECT / "notebooks/00_instance_setup_and_recovery.ipynb", as_version=4)
upstream_exports = [cell.source for cell in upstream.cells if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", [])]
for index, source in enumerate(upstream_exports):
    exec(compile(source, f"00:ahr-export-{index}", "exec"), globals())
RECOVERY_SOURCE_SHA256 = hashlib.sha256("\n".join(upstream_exports).encode()).hexdigest()
THEORY_SHA256 = file_sha256(THEORY_PATH)
started_at = time.monotonic()
print("Theory:", THEORY_PATH.name, "; SHA256:", THEORY_SHA256[:16])
print("Loaded recovery definitions only; no upstream workflows executed.")

## 2. Export configuration, masking and pheromone contracts

Inputs are numeric tokens `x[batch,tokens,16]` plus Boolean validity and candidate masks. Tokens include complete task information supplied by the caller; the generic model does not infer CSP or Go semantics. The synthetic adapter later provides a complete, explicitly bounded Boolean truth relation, partial assignment, and proposed branch.

`iterations` counts additional **Queen cycles** (`K*T` microsteps each). `micro_steps` permits interruption inside a Scout loop; do not supply both arguments. State includes Q/W/U, pheromone, partial summaries, loop counters and a fingerprint binding it to the exact batch. New calls without state begin a new problem. State is never hidden inside the module.

Continuation requires the same model weights and configuration. The state checks configuration and input identity; checkpoint restore additionally verifies all weights. From a mid-loop state, one requested Queen cycle means exactly `K*T` further microsteps, preserving the starting offset.

In [ ]:
from dataclasses import dataclass, asdict
from collections.abc import Mapping
import math, hashlib, json
import torch
from torch import nn

AHR_IMPLEMENTATION = "unified-theory-continuous-v2"

@dataclass(frozen=True)
class AHRConfig:
    input_dim: int = 16
    hidden_dim: int = 32
    iterations: int = 2
    worker_steps: int = 2
    scout_steps: int = 3
    max_iterations: int = 8
    max_tokens: int = 256
    dropout: float = 0.1
    evaporation: float = 0.2
    pheromone_min: float = -2.0
    pheromone_max: float = 2.0
    pheromone_strength: float = 1.0
    deposit_scale: float = 0.1

    def __post_init__(self):
        for name in ("input_dim", "hidden_dim", "iterations", "worker_steps", "scout_steps", "max_iterations", "max_tokens"):
            if type(getattr(self, name)) is not int or getattr(self, name) < 1:
                raise ValueError(f"{name} must be a positive integer")
        if self.hidden_dim < 2 or self.iterations > self.max_iterations:
            raise ValueError("Require hidden_dim >= 2 and iterations <= max_iterations")
        if self.max_iterations * self.worker_steps * self.scout_steps > 512:
            raise ValueError("This architecture validation supports at most 512 microsteps")
        for name in ("dropout", "evaporation", "pheromone_min", "pheromone_max", "pheromone_strength", "deposit_scale"):
            value = getattr(self, name)
            if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value):
                raise ValueError(f"{name} must be finite")
        if not 0 <= self.dropout < 1 or not 0 < self.evaporation <= 1:
            raise ValueError("Require dropout in [0,1) and evaporation in (0,1]")
        if not self.pheromone_min <= 0 <= self.pheromone_max or self.pheromone_min == self.pheromone_max:
            raise ValueError("Pheromone bounds must contain initial zero and have positive width")
        if self.pheromone_strength < 0 or self.deposit_scale <= 0:
            raise ValueError("Require nonnegative prior strength and positive deposit scale")


def detach_ahr_state(state, device=None):
    return {key: value.detach().clone().to(device) if isinstance(value, torch.Tensor) else value for key, value in state.items()}


def pheromone_update(previous, deposit, config, valid_mask):
    updated = torch.clamp((1 - config.evaporation) * previous + deposit,
                          min=config.pheromone_min, max=config.pheromone_max)
    return torch.where(valid_mask, updated, torch.zeros_like(updated))


def pheromone_policy(log_preferences, pheromone, allowed_mask, strength=1.0):
    if allowed_mask.dtype != torch.bool or allowed_mask.shape != log_preferences.shape or pheromone.shape != log_preferences.shape:
        raise ValueError("Prior inputs must share a shape; allowed_mask must be Boolean")
    if not bool(allowed_mask.any(-1).all()):
        raise ValueError("An empty candidate set must abstain outside the neural policy")
    scores = (log_preferences + strength * pheromone).masked_fill(~allowed_mask, -torch.inf)
    return scores.float().softmax(-1).to(log_preferences.dtype)


def _ahr_batch_digest(batch):
    digest = hashlib.sha256()
    for name in ("x", "valid_mask", "action_mask"):
        value = batch[name]
        if name == "x":
            value = torch.where(batch["valid_mask"].unsqueeze(-1), value, torch.zeros_like(value))
        value = value.detach().cpu().contiguous()
        digest.update(str((name, tuple(value.shape), value.dtype)).encode())
        digest.update(value.reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()

## 3. Export learned role updates and canonical stigmergy

Scout proposals attend to the task using the current Q/W/U and the pheromone prior. Worker consumes a typed mean summary of its `T` Scout states; Queen consumes a mean summary of its `K` Worker states. `Deposit` sees Q/W/U **before** the Scout update, as in Eq. 4.2. All roles have explicit finite energy heads; those scores never certify correctness.

In [ ]:
class RoleEnergy(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(2 * d, d), nn.Tanh(), nn.Linear(d, 1))

    def forward(self, tokens, role):
        return self.network(torch.cat((tokens, role[:, None, :].expand_as(tokens)), -1)).squeeze(-1)


class Scout(nn.Module):
    def __init__(self, config):
        super().__init__()
        d = config.hidden_dim
        self.query = nn.Linear(3 * d, d)
        self.keys = nn.Linear(d, d, bias=False)
        self.context = nn.Linear(3 * d, d)
        self.recurrent = nn.GRUCell(d, d)
        self.dropout = nn.Dropout(config.dropout)
        self.energy = RoleEnergy(d)
        self.strength = config.pheromone_strength

    def forward(self, tokens, queen, worker, previous, pheromone, valid_mask):
        query = self.query(torch.cat((queen, worker, previous), -1))
        logits = torch.einsum("bd,bld->bl", query, self.keys(tokens)) / math.sqrt(tokens.shape[-1])
        weights = pheromone_policy(logits, pheromone, valid_mask, self.strength)
        attended = torch.einsum("bl,bld->bd", weights.to(tokens.dtype), tokens)
        context = torch.tanh(self.context(torch.cat((attended, queen, worker), -1)))
        return self.recurrent(self.dropout(context), previous), weights


class Worker(nn.Module):
    def __init__(self, config):
        super().__init__()
        d = config.hidden_dim
        self.context = nn.Linear(4 * d, d)
        self.recurrent = nn.GRUCell(d, d)
        self.dropout = nn.Dropout(config.dropout)
        self.energy = RoleEnergy(d)

    def forward(self, previous, queen, scouts, memory_context, task_context):
        context = torch.tanh(self.context(torch.cat((queen, scouts, memory_context, task_context), -1)))
        return self.recurrent(self.dropout(context), previous)


class Queen(nn.Module):
    def __init__(self, config):
        super().__init__()
        d = config.hidden_dim
        self.context = nn.Linear(3 * d, d)
        self.recurrent = nn.GRUCell(d, d)
        self.energy = RoleEnergy(d)

    def forward(self, previous, workers, memory_context, task_context):
        context = torch.tanh(self.context(torch.cat((workers, memory_context, task_context), -1)))
        return self.recurrent(context, previous)


class StigmergicMemory(nn.Module):
    def __init__(self, config):
        super().__init__()
        d = config.hidden_dim
        self.config = config
        self.deposit = nn.Sequential(nn.Linear(4 * d, d), nn.Tanh(), nn.Linear(d, 1), nn.Tanh())

    def forward(self, tokens, queen, worker, scout, previous, valid_mask):
        contexts = [role[:, None, :].expand_as(tokens) for role in (queen, worker, scout)]
        deposit = self.config.deposit_scale * self.deposit(torch.cat((tokens, *contexts), -1)).squeeze(-1)
        deposit = torch.where(valid_mask, deposit, torch.zeros_like(deposit)).to(previous.dtype)
        return pheromone_update(previous, deposit, self.config, valid_mask), deposit

    def read(self, tokens, pheromone, valid_mask):
        weights = pheromone_policy(torch.zeros_like(pheromone), pheromone, valid_mask, self.config.pheromone_strength)
        return torch.einsum("bl,bld->bd", weights.to(tokens.dtype), tokens)

## 4. Export nested dynamics and explicit resumable state

Resets occur lazily at the beginning of each actual Queen/Worker interval. This makes the counts exact: `N*K*T` Scout/deposit updates, `N*K` Worker updates, `N` Queen updates, `N` Worker resets and `N*K` Scout resets. No unnecessary reset is performed after the last requested microstep. Pheromone is carried across every loop boundary.

In [ ]:
class NativeAHR(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = AHRConfig() if config is None else config
        if not isinstance(self.config, AHRConfig):
            raise TypeError("config must be AHRConfig")
        d = self.config.hidden_dim
        self.encoder = nn.Sequential(nn.Linear(self.config.input_dim, d), nn.LayerNorm(d), nn.Tanh())
        self.initial_queen = nn.Linear(d, d)
        self.reset_worker = nn.Linear(2 * d, d)
        self.reset_scout = nn.Linear(3 * d, d)
        self.queen = Queen(self.config)
        self.worker = Worker(self.config)
        self.scout = Scout(self.config)
        self.memory = StigmergicMemory(self.config)
        self.config_digest = hashlib.sha256((AHR_IMPLEMENTATION + json.dumps(asdict(self.config), sort_keys=True)).encode()).hexdigest()

    def _validate_batch(self, batch):
        if not isinstance(batch, Mapping) or set(batch) != {"x", "valid_mask", "action_mask"}:
            raise ValueError("Batch must contain exactly x, valid_mask, action_mask")
        x, valid, actions = (batch[k] for k in ("x", "valid_mask", "action_mask"))
        if not all(isinstance(v, torch.Tensor) for v in (x, valid, actions)):
            raise TypeError("Batch values must be tensors")
        reference = next(self.parameters())
        if x.ndim != 3 or x.shape[0] < 1 or not 1 <= x.shape[1] <= self.config.max_tokens or x.shape[-1] != self.config.input_dim:
            raise ValueError("Invalid token batch dimensions")
        if x.dtype != reference.dtype or x.device != reference.device or not x.is_floating_point():
            raise ValueError("Input must match model floating dtype/device; autocast is supported")
        for mask in (valid, actions):
            if mask.dtype != torch.bool or mask.shape != x.shape[:2] or mask.device != x.device:
                raise ValueError("Masks must be Boolean [batch,tokens] on the input device")
        if not bool(valid.any(-1).all()) or not bool(actions.any(-1).all()) or bool((actions & ~valid).any()):
            raise ValueError("Each example needs visible tokens and a nonempty subset of candidate actions")
        if not bool(torch.isfinite(x[valid]).all()):
            raise ValueError("Visible input features must be finite")
        return x, valid, actions

    def _validate_state(self, state, batch, signature):
        x = batch["x"]
        b, length = x.shape[:2]
        d = self.config.hidden_dim
        roles = {"queen", "worker", "scout", "scout_summary", "worker_summary"}
        counters = {"iteration", "worker_index", "scout_index", "micro_step"}
        if not isinstance(state, Mapping) or set(state) != roles | counters | {"pheromone", "input_digest", "config_digest"}:
            raise ValueError("Recurrent state is missing or contains unexpected fields")
        if state["config_digest"] != self.config_digest:
            raise ValueError("State belongs to a different architecture configuration")
        if state["input_digest"] != signature:
            raise ValueError("State belongs to a different task batch or sample order")
        for name in roles | {"pheromone"}:
            value = state[name]
            shape = (b, length) if name == "pheromone" else (b, d)
            if not isinstance(value, torch.Tensor) or tuple(value.shape) != shape or value.device != x.device or value.dtype != x.dtype:
                raise ValueError(f"Invalid state tensor {name}")
            if not bool(torch.isfinite(value).all()):
                raise ValueError(f"Nonfinite recurrent state {name}")
        if any(type(state[key]) is not int or state[key] < 0 for key in counters):
            raise ValueError("Loop counters must be nonnegative integers")
        n, k, t = (state[key] for key in ("iteration", "worker_index", "scout_index"))
        c = self.config
        if n > c.max_iterations or k >= c.worker_steps or t >= c.scout_steps or state["micro_step"] != n * c.worker_steps * c.scout_steps + k * c.scout_steps + t:
            raise ValueError("Inconsistent nested-loop counters")
        p = state["pheromone"]
        if bool(((p < c.pheromone_min) | (p > c.pheromone_max)).any()) or bool((p[~batch["valid_mask"]] != 0).any()):
            raise ValueError("Pheromone violates declared bounds or padding")

    def forward(self, batch, iterations=None, micro_steps=None, state=None):
        x, valid, actions = self._validate_batch(batch)
        c = self.config
        if iterations is not None and micro_steps is not None:
            raise ValueError("Choose Queen cycles or microsteps, not both")
        cycles = c.iterations if iterations is None else iterations
        if micro_steps is None:
            if type(cycles) is not int or cycles < 1:
                raise ValueError("iterations must be a positive integer")
            count = cycles * c.worker_steps * c.scout_steps
        else:
            if type(micro_steps) is not int or micro_steps < 1:
                raise ValueError("micro_steps must be a positive integer")
            count = micro_steps
        signature = _ahr_batch_digest(batch)
        tokens = self.encoder(torch.where(valid.unsqueeze(-1), x, torch.zeros_like(x)))
        tokens = torch.where(valid.unsqueeze(-1), tokens, torch.zeros_like(tokens))
        pooled = tokens.sum(1) / valid.sum(1, keepdim=True)
        if state is None:
            queen = torch.tanh(self.initial_queen(pooled)).to(x.dtype)
            current = {"queen": queen, "worker": torch.zeros_like(queen), "scout": torch.zeros_like(queen),
                       "scout_summary": torch.zeros_like(queen), "worker_summary": torch.zeros_like(queen),
                       "pheromone": torch.zeros_like(x[:, :, 0]), "iteration": 0, "worker_index": 0,
                       "scout_index": 0, "micro_step": 0, "input_digest": signature, "config_digest": self.config_digest}
        else:
            self._validate_state(state, batch, signature)
            current = dict(state)  # All tensor updates below are out of place.
        if current["micro_step"] + count > c.max_iterations * c.worker_steps * c.scout_steps:
            raise ValueError("Requested continuation exceeds the total microstep budget")
        trace = []
        for _ in range(count):
            q, w, u = (current[key] for key in ("queen", "worker", "scout"))
            n, k, t = (current[key] for key in ("iteration", "worker_index", "scout_index"))
            worker_reset, scout_reset = (k == 0 and t == 0), (t == 0)
            scout_summary, worker_summary = current["scout_summary"], current["worker_summary"]
            if worker_reset:
                w = torch.tanh(self.reset_worker(torch.cat((q, pooled), -1))).to(x.dtype)
                worker_summary = torch.zeros_like(q)
            if scout_reset:
                u = torch.tanh(self.reset_scout(torch.cat((q, w, pooled), -1))).to(x.dtype)
                scout_summary = torch.zeros_like(q)
            old_pheromone = current["pheromone"]
            # Both equations use the same pre-update q,w,u,rho (PDF Eq.4.1 and Eq.4.2).
            new_u, attention = self.scout(tokens, q, w, u, old_pheromone, valid)
            pheromone, deposit = self.memory(tokens, q, w, u, old_pheromone, valid)
            new_u = new_u.to(x.dtype)
            scout_summary = scout_summary + new_u
            t += 1
            worker_updated = t == c.scout_steps
            queen_updated = False
            if worker_updated:
                memory_context = self.memory.read(tokens, pheromone, valid)
                w = self.worker(w, q, scout_summary / c.scout_steps, memory_context, pooled).to(x.dtype)
                worker_summary = worker_summary + w
                k += 1
                t = 0
                if k == c.worker_steps:
                    q = self.queen(q, worker_summary / c.worker_steps, memory_context, pooled).to(x.dtype)
                    n += 1
                    k = 0
                    queen_updated = True
            current = {"queen": q, "worker": w, "scout": new_u, "pheromone": pheromone,
                       "scout_summary": scout_summary, "worker_summary": worker_summary,
                       "iteration": n, "worker_index": k, "scout_index": t,
                       "micro_step": current["micro_step"] + 1, "input_digest": signature, "config_digest": self.config_digest}
            trace.append({"scout_update": True, "worker_update": worker_updated, "queen_update": queen_updated,
                          "worker_reset": worker_reset, "scout_reset": scout_reset,
                          "attention": attention, "pheromone_before": old_pheromone,
                          "deposit": deposit, "pheromone_after": pheromone,
                          "iteration": n, "worker_index": k, "scout_index": t})
        eq = self.queen.energy(tokens, current["queen"])
        ew = self.worker.energy(tokens, current["worker"])
        eu = self.scout.energy(tokens, current["scout"])
        ep = -c.pheromone_strength * current["pheromone"]
        soft = eq + ew + eu + ep
        if not bool(torch.isfinite(torch.stack((eq.float(), ew.float(), eu.float(), ep.float(), soft.float()))).all()):
            raise FloatingPointError("Nonfinite learned energy; no ranking or acceptance is permitted")
        hard = torch.where(actions, torch.zeros_like(soft), torch.full_like(soft, torch.inf))
        return {"policy_logits": -(soft + hard), "energies": {"queen": eq, "worker": ew, "scout": eu,
                 "pheromone": ep, "soft": soft}, "hard_action_energy": hard,
                "state": current, "trace": trace, "verified_status": "UNKNOWN"}

## 5. Synthetic fixtures and deterministic setup

The diagnostic loss below exercises finite role energies and candidate probabilities. It is used for backward checks only; no optimizer is constructed. No dataset labels or final-test files are loaded.

In [ ]:
import random, numpy as np
from datetime import datetime, timezone

SEED = 20260923
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
CONFIG = AHRConfig()
features = torch.sin(torch.arange(2 * 7 * CONFIG.input_dim, dtype=torch.float32).reshape(2, 7, CONFIG.input_dim) / 11)
valid = torch.tensor([[True] * 7, [True] * 4 + [False] * 3])
actions = valid.clone()
actions[:, 0] = False
BATCH = {"x": features, "valid_mask": valid, "action_mask": actions}
model = NativeAHR(CONFIG).eval()
INITIAL_WEIGHTS = copy.deepcopy(model.state_dict())
PARAMETER_COUNT = sum(p.numel() for p in model.parameters())
checks = {}

def close(left, right, atol=3e-6, rtol=3e-5):
    torch.testing.assert_close(left, right, atol=atol, rtol=rtol)

def diagnostic_loss(output, batch):
    selected = batch["action_mask"].to(torch.int64).argmax(-1)
    policy_loss = -output["policy_logits"].log_softmax(-1).gather(1, selected[:, None]).mean()
    energy_loss = sum(output["energies"][role][batch["valid_mask"]].square().mean() for role in ("queen", "worker", "scout"))
    return policy_loss + energy_loss + output["state"]["pheromone"].square().mean()

print(f"{PARAMETER_COUNT:,} random parameters; N={CONFIG.iterations}, K={CONFIG.worker_steps}, T={CONFIG.scout_steps}")
print("Torch:", torch.__version__, "; CUDA devices:", torch.cuda.device_count())

## 6. Equation, masking and state checks

Verify the exact role clocks and resets, the pre-update deposition equation, pheromone carry/clamp/prior, input isolation, padded NaNs, candidate masking, configuration binding and split recurrence. These tests exercise the stated mechanisms, not learned performance.

In [ ]:
with torch.no_grad():
    baseline = model(BATCH)
    expected_counts = {"scout_update": 12, "worker_update": 4, "queen_update": 2, "worker_reset": 2, "scout_reset": 4}
    counts = {name: sum(step[name] for step in baseline["trace"]) for name in expected_counts}
    assert counts == expected_counts
    assert baseline["state"]["micro_step"] == 12 and baseline["state"]["iteration"] == 2
    prior_pheromone = torch.zeros_like(features[:, :, 0])
    for step in baseline["trace"]:
        close(step["pheromone_before"], prior_pheromone, atol=0, rtol=0)
        expected = pheromone_update(prior_pheromone, step["deposit"], CONFIG, valid)
        close(step["pheromone_after"], expected, atol=0, rtol=0)
        assert torch.all(step["pheromone_after"] >= CONFIG.pheromone_min)
        assert torch.all(step["pheromone_after"] <= CONFIG.pheromone_max)
        assert torch.count_nonzero(step["attention"][~valid]) == 0
        close(step["attention"].sum(-1), torch.ones(2))
        prior_pheromone = expected
    # Independently reproduce the very first deposit from pre-update Q/W/U.
    clean = torch.where(valid.unsqueeze(-1), features, torch.zeros_like(features))
    tokens = model.encoder(clean)
    tokens = torch.where(valid.unsqueeze(-1), tokens, torch.zeros_like(tokens))
    pooled = tokens.sum(1) / valid.sum(1, keepdim=True)
    q0 = torch.tanh(model.initial_queen(pooled))
    w0 = torch.tanh(model.reset_worker(torch.cat((q0, pooled), -1)))
    u0 = torch.tanh(model.reset_scout(torch.cat((q0, w0, pooled), -1)))
    _, deposit0 = model.memory(tokens, q0, w0, u0, torch.zeros_like(valid, dtype=torch.float32), valid)
    close(baseline["trace"][0]["deposit"], deposit0, atol=0, rtol=0)
    # Published two-action example: softmax([log 3,0] + log2*[0,1]) = [3/5,2/5].
    probability = pheromone_policy(torch.tensor([[math.log(3), 0.0]]), torch.tensor([[0.0, 1.0]]), torch.ones(1, 2, dtype=torch.bool), math.log(2))
    close(probability, torch.tensor([[0.6, 0.4]]))
    unit_mask = torch.ones(1, 2, dtype=torch.bool)
    old = torch.tensor([[1.0, -1.0]])
    close(pheromone_update(old, torch.zeros_like(old), CONFIG, unit_mask), old * (1-CONFIG.evaporation), atol=0, rtol=0)
    close(pheromone_update(old, torch.tensor([[100.0, -100.0]]), CONFIG, unit_mask), torch.tensor([[CONFIG.pheromone_max, CONFIG.pheromone_min]]), atol=0, rtol=0)
    alternate = {**BATCH, "x": features.clone()}
    alternate["x"][~valid] = torch.nan
    close(model(alternate)["policy_logits"], baseline["policy_logits"], atol=0, rtol=0)
    extended = {"x": torch.cat((features, torch.full((2, 3, CONFIG.input_dim), torch.nan)), 1),
                "valid_mask": torch.cat((valid, torch.zeros(2, 3, dtype=torch.bool)), 1),
                "action_mask": torch.cat((actions, torch.zeros(2, 3, dtype=torch.bool)), 1)}
    padded = model(extended)
    close(padded["policy_logits"][:, :7], baseline["policy_logits"])
    for role in ("queen", "worker", "scout"):
        close(padded["state"][role], baseline["state"][role])
    for index, length in enumerate(valid.sum(-1).tolist()):
        single = {key: value[index:index+1, :length] for key, value in BATCH.items()}
        close(model(single)["policy_logits"], baseline["policy_logits"][index:index+1, :length])
    assert (baseline["policy_logits"].softmax(-1)[~actions] == 0).all()
    assert torch.isinf(baseline["hard_action_energy"][~actions]).all()
    partial = model(BATCH, micro_steps=1)
    partial_copy = detach_ahr_state(partial["state"])
    continued = model(BATCH, micro_steps=11, state=partial["state"])
    close(continued["policy_logits"], baseline["policy_logits"], atol=0, rtol=0)
    for key in partial_copy:
        if isinstance(partial_copy[key], torch.Tensor):
            close(partial["state"][key], partial_copy[key], atol=0, rtol=0)
            close(continued["state"][key], baseline["state"][key], atol=0, rtol=0)
    assert [step["queen_update"] for step in partial["trace"] + continued["trace"]] == [step["queen_update"] for step in baseline["trace"]]
    other_output = model({**BATCH, "x": -features})
    assert not torch.allclose(other_output["energies"]["soft"], baseline["energies"]["soft"])
    close(model(BATCH)["policy_logits"], baseline["policy_logits"], atol=0, rtol=0)
    config_deeper = AHRConfig(iterations=3)
    deeper = NativeAHR(config_deeper)
    assert sum(p.numel() for p in deeper.parameters()) == PARAMETER_COUNT
checks["theory_mechanisms"] = {"role_counts": counts, "pre_update_deposit": True, "clamp_and_carry": True,
                              "published_prior_example": [0.6, 0.4], "padding_and_sample_isolation": True,
                              "mid_scout_split_bitwise": True, "parameters_independent_of_depth": True}

invalid_cases = []
def reject(name, operation):
    try:
        operation()
    except (ValueError, TypeError, FloatingPointError):
        invalid_cases.append(name)
    else:
        raise AssertionError(f"Invalid input accepted: {name}")
for count in (True, 0, -1, 1.5, 10000):
    reject(f"micro_steps_{count}", lambda count=count: model(BATCH, micro_steps=count))
reject("two_step_units", lambda: model(BATCH, micro_steps=1, iterations=1))
for name, bad in (("integer_input", {**BATCH, "x": features.long()}),
                  ("wrong_dimension", {**BATCH, "x": features[:, :, :-1]}),
                  ("float_mask", {**BATCH, "valid_mask": valid.float()}),
                  ("all_masked", {**BATCH, "valid_mask": torch.zeros_like(valid)}),
                  ("empty_actions", {**BATCH, "action_mask": torch.zeros_like(actions)}),
                  ("padded_action", {**BATCH, "action_mask": torch.ones_like(actions)}),
                  ("target_leak", {**BATCH, "target": 1})):
    reject(name, lambda bad=bad: model(bad))
for name, bad in (("cursor", {**partial_copy, "micro_step": 2}),
                  ("configuration", {**partial_copy, "config_digest": "wrong"}),
                  ("shape", {**partial_copy, "scout": partial_copy["scout"][:1]}),
                  ("nan", {**partial_copy, "queen": partial_copy["queen"] * torch.nan}),
                  ("pheromone_bound", {**partial_copy, "pheromone": partial_copy["pheromone"] + 100})):
    reject(name, lambda bad=bad: model(BATCH, micro_steps=1, state=bad))
reject("other_task_state", lambda: model({**BATCH, "x": -features}, micro_steps=1, state=partial_copy))
for kwargs in ({"hidden_dim": 1}, {"evaporation": 0}, {"evaporation": 2}, {"dropout": float("nan")}, {"scout_steps": True}, {"max_iterations": 1000}):
    reject(str(kwargs), lambda kwargs=kwargs: AHRConfig(**kwargs))
checks["invalid_input_rejections"] = invalid_cases

# A terminal diagnostic gradient must reach an earlier explicit Q/W/U/rho state.
leaves = {key: value.clone().requires_grad_(True) if isinstance(value, torch.Tensor) else value for key, value in partial_copy.items()}
model.zero_grad(set_to_none=True)
diagnostic_loss(model(BATCH, micro_steps=11, state=leaves), BATCH).backward()
for key in ("queen", "worker", "scout", "pheromone"):
    assert leaves[key].grad is not None and torch.isfinite(leaves[key].grad).all() and leaves[key].grad.abs().sum() > 0
model.zero_grad(set_to_none=True)
checks["gradient_through_explicit_state"] = True
print("Nested dynamics:", counts)
print("Mask/state/prior checks PASS; invalid cases rejected:", len(invalid_cases))

## 7. CPU and every available GPU: gradients and serialization

Each device runs the same untrained weights. Require finite nonzero gradients in every parameter tensor, exact same-device weight save/load, unchanged weights after backward, and bfloat16 forward/backward on CUDA. GPUs are checked individually; this is not distributed training validation.

In [ ]:
import io

def device_architecture_check(device):
    candidate = NativeAHR(CONFIG).to(device)
    candidate.load_state_dict(INITIAL_WEIGHTS)
    batch = {key: value.to(device) for key, value in BATCH.items()}
    before = {key: value.detach().cpu().clone() for key, value in candidate.state_dict().items()}
    candidate.eval()
    with torch.no_grad():
        output = candidate(batch)
        close(output["policy_logits"].cpu(), baseline["policy_logits"], atol=3e-5, rtol=3e-4)
        buffer = io.BytesIO()
        torch.save({"config": asdict(CONFIG), "model": candidate.state_dict()}, buffer)
        buffer.seek(0)
        saved = torch.load(buffer, map_location=device, weights_only=True)
        restored = NativeAHR(AHRConfig(**saved["config"])).to(device).eval()
        restored.load_state_dict(saved["model"], strict=True)
        close(restored(batch)["policy_logits"], output["policy_logits"], atol=0, rtol=0)
    candidate.train()  # Dropout enabled for validation, no optimization.
    candidate.zero_grad(set_to_none=True)
    loss = diagnostic_loss(candidate(batch), batch)
    loss.backward()
    gradient_norms = {}
    for name, parameter in candidate.named_parameters():
        assert parameter.grad is not None and torch.isfinite(parameter.grad).all(), f"Invalid gradient {device}/{name}"
        assert parameter.grad.abs().sum() > 0, f"Unused parameter {device}/{name}"
        component = name.split('.')[0]
        gradient_norms[component] = gradient_norms.get(component, 0.0) + float(parameter.grad.detach().float().norm().cpu())
    autocast_checked = False
    if str(device).startswith("cuda"):
        candidate.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            mixed = candidate(batch)
            mixed_loss = diagnostic_loss(mixed, batch)
        assert torch.isfinite(mixed_loss)
        mixed_loss.backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in candidate.parameters())
        candidate.eval()
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            first = candidate(batch, micro_steps=1)
            result = candidate(batch, micro_steps=2, state=first["state"])
            assert torch.isfinite(result["energies"]["soft"]).all()
        torch.cuda.synchronize(device)
        autocast_checked = True
    assert all(torch.equal(value.cpu(), before[name]) for name, value in candidate.state_dict().items())
    return {"device": str(device), "passed": True, "parameter_tensors_with_finite_nonzero_gradient": len(list(candidate.parameters())),
            "gradient_norms": gradient_norms, "same_device_save_load_bitwise": True, "weights_unchanged": True,
            "bfloat16_autocast": autocast_checked, "optimizer_steps": 0}

DEVICE_CHECKS = [device_architecture_check(device) for device in ["cpu"] + [f"cuda:{i}" for i in range(torch.cuda.device_count())]]
checks["devices"] = DEVICE_CHECKS
for item in DEVICE_CHECKS:
    print(item["device"], "PASS:", item["parameter_tensors_with_finite_nonzero_gradient"], "gradient-bearing parameter tensors; 0 updates")

## 8. Typed QWS compatibility, strict transitions and hard energy

The neural core is now connected to a finite, inspectable AHR instantiation. The controller's Queen has explicit budget/verification/stopping fields; Worker records the actual partial construction; Scout records its frontier and unexplored branches. Learned scores select among these authorized actions. All six compatibility clauses are hard predicates. The trusted transition computes state changes and certificate checks itself.

The single goal `solve`, Boolean assignment rules and fixed next-variable relevance are explicit choices for this small domain. Finite Scout novelty/uncertainty estimates are constant singleton grids in this fixture; learned novelty/coverage claims are not made. The neural recurrence uses continuous representations and does not inherit a finite-state existence theorem.

### Finite QWS controller and independent verification

This architecture smoke fixture instantiates the six clauses on a synthetic Boolean CSP with at most three variables. It uses no dataset, external solver, optimizer or training. Search explores only typed Scout proposals, Queen allocation and Worker assignment/backtracking transitions. Learned finite role scores order available actions; they never establish correctness. A budget-limited result is UNKNOWN, and this prototype makes no UNSAT or global minimum-energy claim.

Source map: PDF physical pp. 25–30 (role coordinates, six QWS clauses and verification contract), pp. 36–40 (eight energy terms and bounded role-factored search), pp. 67–68 (CSP verifier and role mapping), p. 85 (unified definition). Singleton coordinates retain explicit typed projections. The neural recurrence defined elsewhere is a continuous realization; this small symbolic controller supplies the discrete governance and verification boundary.

Trust boundary: the CSP, initial-state constructor, compatibility predicates, strict transition function and exact membership checker are trusted. The neural scorer receives immutable problem/assignment/action values and returns only finite energy triples. It cannot supply acceptance, pruning or verification evidence. Python is not a security sandbox: arbitrary code replacing these trusted definitions is outside this contract. A stored pass is bound to the complete problem digest and the entire Worker state; construction changes clear it, and the status reader independently checks the current certificate.

Run-time limits are cooperative host checks before and after model scoring and before acceptance. Python/OS scheduling and a blocked callback cannot provide a hard real-time guarantee. `max_expansions` counts every typed transition, including verification and acceptance. One remaining transition is reserved for justified abstention. A zero budget or elapsed deadline returns UNKNOWN without inventing a transition. The checker can reject a certificate but never launches search.

In [ ]:

from dataclasses import dataclass, field, replace
from enum import Enum
from typing import Callable, Optional
import hashlib
import json
import math
import time

class AHRStatus(str, Enum):
    SAT = "SAT"
    UNKNOWN = "UNKNOWN"

class AHRDecision(str, Enum):
    UNDECIDED = "undecided"
    ACCEPT = "accept"
    REJECT = "reject"
    ABSTAIN = "abstain"

@dataclass(frozen=True, order=True)
class Branch:
    variable: int
    value: int

    def __post_init__(self):
        if type(self.variable) is not int or self.variable < 0:
            raise ValueError("Branch variable must be a nonnegative integer")
        if type(self.value) is not int or self.value not in (0, 1):
            raise ValueError("Branch value must be the integer 0 or 1")

@dataclass(frozen=True)
class BinaryCSP:
    n_vars: int
    allowed_assignments: tuple[tuple[int, ...], ...]
    name: str = "synthetic"

    def __post_init__(self):
        if type(self.n_vars) is not int or not 1 <= self.n_vars <= 3:
            raise ValueError("Architecture fixtures require 1..3 Boolean variables")
        values = tuple(tuple(row) for row in self.allowed_assignments)
        if any(len(row) != self.n_vars or any(type(v) is not int or v not in (0, 1) for v in row) for row in values):
            raise ValueError("Allowed rows must be complete Boolean assignments")
        object.__setattr__(self, "allowed_assignments", tuple(sorted(set(values))))

    @property
    def digest(self):
        payload = json.dumps((self.n_vars, self.allowed_assignments, self.name), separators=(",", ":"))
        return hashlib.sha256(payload.encode()).hexdigest()

@dataclass(frozen=True)
class QueenState:
    strategy: str = "finite_role_energy_order"
    budget_remaining: int = 0
    verification_policy: str = "required"
    acceptance_threshold: int = 0
    stopping_rule: str = "budget_or_verified_failure"
    arbitration_memory: tuple[str, ...] = ()

@dataclass(frozen=True)
class WorkerState:
    active_subgoal: str
    local_construction: tuple[Optional[int], ...]
    domain_rule: str = "assign_boolean"
    derivation: tuple[Branch, ...] = ()
    local_feedback: str = "construction"
    progress: int = 0

@dataclass(frozen=True)
class ScoutState:
    frontier: frozenset[Branch] = frozenset()
    hypotheses: frozenset[Branch] = frozenset()
    unexplored: frozenset[Branch] = frozenset()
    energy: tuple[tuple[Branch, int], ...] = ()
    novelty: tuple[tuple[Branch, int], ...] = ()
    uncertainty: tuple[tuple[Branch, int], ...] = ()

@dataclass(frozen=True)
class _VerifierRecord:
    problem_digest: str
    worker_digest: str
    status: str
    checked_at_step: int

@dataclass(frozen=True)
class Continuation:
    worker: WorkerState
    scout: ScoutState

@dataclass(frozen=True)
class TaskState:
    selected_subgoals: frozenset[str] = frozenset({"solve"})
    verification_record: Optional[_VerifierRecord] = None
    continuations: tuple[Continuation, ...] = ()
    global_step: int = 0
    terminal_decision: AHRDecision = AHRDecision.UNDECIDED

@dataclass(frozen=True)
class AHRState:
    queen: QueenState
    worker: WorkerState
    scout: ScoutState
    task: TaskState

@dataclass(frozen=True)
class QueenAction:
    select: frozenset[str] = frozenset()
    prune: frozenset[Branch] = frozenset()
    verify: bool = False
    decision: AHRDecision = AHRDecision.UNDECIDED
    stop: bool = False

@dataclass(frozen=True)
class WorkerAction:
    subgoals: frozenset[str] = frozenset()
    rule: Optional[str] = None
    use: tuple[Branch, ...] = ()

@dataclass(frozen=True)
class ScoutAction:
    propose: frozenset[Branch] = frozenset()

@dataclass(frozen=True)
class AHRAction:
    queen: QueenAction = field(default_factory=QueenAction)
    worker: WorkerAction = field(default_factory=WorkerAction)
    scout: ScoutAction = field(default_factory=ScoutAction)

@dataclass(frozen=True)
class AHRTransition:
    before: AHRState
    action: AHRAction
    after: AHRState
    role_energies: tuple[float, float, float]

@dataclass(frozen=True)
class AHRSearchResult:
    status: AHRStatus
    assignment: Optional[tuple[int, ...]]
    trace: tuple[AHRTransition, ...]
    expansions: int
    elapsed_s: float
    energy_breakdown: dict[str, float]
    reason: str
    initial_state: AHRState
    final_state: AHRState


def _worker_digest(worker):
    return hashlib.sha256(repr(worker).encode()).hexdigest()


def verify_binary_assignment(problem, assignment):
    """Independent direct membership check: no ranking, model call or search."""
    return (len(assignment) == problem.n_vars
            and all(type(v) is int and v in (0, 1) for v in assignment)
            and tuple(assignment) in problem.allowed_assignments)


def _complete(worker):
    return all(v is not None for v in worker.local_construction)


def verifier_status(problem, state):
    """A pass applies only to this exact problem and Worker certificate."""
    record = state.task.verification_record
    if record is None or record.problem_digest != problem.digest or record.worker_digest != _worker_digest(state.worker):
        return "unchecked"
    actual = "pass" if verify_binary_assignment(problem, state.worker.local_construction) else "fail"
    return record.status if record.status == actual else "unchecked"


def _relevant(problem, worker):
    partial = worker.local_construction
    if len(partial) != problem.n_vars or _complete(worker):
        return frozenset()
    variable = next(i for i, value in enumerate(partial) if value is None)
    return frozenset(Branch(variable, v) for v in (0, 1))


def _scout(problem, worker, frontier=frozenset(), unexplored=None):
    relevant = _relevant(problem, worker)
    unexplored = relevant if unexplored is None else frozenset(unexplored)
    universe = tuple(Branch(i, v) for i in range(problem.n_vars) for v in (0, 1))
    zeros = tuple((branch, 0) for branch in universe)
    return ScoutState(frozenset(frontier), frozenset(frontier), unexplored, zeros, zeros, zeros)


def initial_ahr_state(problem, max_expansions):
    if type(max_expansions) is not int or max_expansions < 0:
        raise ValueError("max_expansions must be a nonnegative integer")
    worker = WorkerState("solve", (None,) * problem.n_vars)
    return AHRState(QueenState(budget_remaining=max_expansions), worker, _scout(problem, worker), TaskState())


def _auth(state):
    return frozenset({"solve"})


def _rules(state):
    rules = {"assign_boolean"}
    if state.task.continuations:
        rules.add("restore_continuation")
    return frozenset(rules)


def _prune_ok(problem, state, branch):
    # Only a local malformed/conflicting branch or a fully constructed rejected
    # certificate establishes pruning. Never solve for a partial completion.
    if branch not in _relevant(problem, state.worker):
        return True
    candidate = list(state.worker.local_construction)
    candidate[branch.variable] = branch.value
    return all(v is not None for v in candidate) and not verify_binary_assignment(problem, tuple(candidate))


def _accept_ok(problem, state, action):
    return (_complete(state.worker)
            and verify_binary_assignment(problem, state.worker.local_construction)
            and action.worker.rule is None
            and not action.worker.use
            and not action.queen.prune
            and not action.scout.propose)


def _exhausted(problem, state):
    if state.task.continuations:
        return False
    if _complete(state.worker):
        return verifier_status(problem, state) == "fail"
    return not (state.scout.frontier | state.scout.unexplored)


def _reject_ok(problem, state, action):
    # Global UNSAT rejection is deliberately unsupported by this architecture smoke.
    return False


def _failure_ok(problem, state, action):
    return state.queen.budget_remaining <= 1 or _exhausted(problem, state)


def qws_clauses(problem, state, action):
    """Exact six-clause hard relation; each boolean is independently observable."""
    q, w, u, s = state.queen, state.worker, state.scout, state.task
    aq, aw, au = action.queen, action.worker, action.scout
    authorized = _auth(state)
    relevant = _relevant(problem, w)
    admissible = relevant
    status = verifier_status(problem, state)
    return {
        "QWS-1": w.active_subgoal in authorized and (aq.select | aw.subgoals) <= authorized,
        "QWS-2": u.frontier <= relevant and (au.propose - aq.prune) <= relevant,
        "QWS-3": set(aw.use) <= ((u.frontier | au.propose) & admissible)
                    and (aw.rule is None or aw.rule in _rules(state)),
        "QWS-4": aq.prune <= (u.frontier | au.propose)
                    and all(_prune_ok(problem, state, branch) for branch in aq.prune)
                    and (aq.decision != AHRDecision.ACCEPT or _accept_ok(problem, state, action))
                    and (aq.decision != AHRDecision.REJECT or _reject_ok(problem, state, action)),
        "QWS-5": not (q.verification_policy == "required" and aq.decision == AHRDecision.ACCEPT)
                    or status == "pass",
        "QWS-6": not aq.stop or (
                    (aq.decision == AHRDecision.ACCEPT and status == "pass")
                    or (aq.decision in (AHRDecision.REJECT, AHRDecision.ABSTAIN)
                        and _failure_ok(problem, state, action))),
    }


def strict_ahr_transition(problem, state, action):
    """Compute all action effects internally; incompatible transitions raise."""
    failed = [name for name, passed in qws_clauses(problem, state, action).items() if not passed]
    if failed:
        raise ValueError("QWS incompatibility: " + ", ".join(failed))
    q, w, u, s = state.queen, state.worker, state.scout, state.task
    aq, aw, au = action.queen, action.worker, action.scout
    if q.budget_remaining <= 0 or s.terminal_decision != AHRDecision.UNDECIDED:
        raise ValueError("No transition after stopping or budget exhaustion")
    if len(w.local_construction) != problem.n_vars or any(v is not None and (type(v) is not int or v not in (0, 1)) for v in w.local_construction):
        raise ValueError("Malformed Worker assignment")
    if aq.verify and (aw.rule is not None or aw.use or aq.decision != AHRDecision.UNDECIDED or aq.stop):
        raise ValueError("Verify on one transition; accept only on a later transition")
    if aq.decision != AHRDecision.UNDECIDED and not aq.stop:
        raise ValueError("Decision must terminate this finite controller")
    if aq.stop and (aw.rule is not None or aw.use or au.propose or aq.prune):
        raise ValueError("Stopping cannot mutate the construction/frontier")
    if set(aw.use) & aq.prune:
        raise ValueError("Worker cannot use a pruned branch")
    if aw.rule is None and aw.use:
        raise ValueError("Branch use requires a Worker rule")
    selected = aq.select if aq.select else s.selected_subgoals
    if selected != frozenset({"solve"}):
        raise ValueError("Selected subgoal must be reflected in the Worker obligation")
    retained = (u.frontier | au.propose) - aq.prune
    unexplored = u.unexplored - au.propose - aq.prune
    next_u = _scout(problem, w, retained, unexplored)
    next_w, stack = w, s.continuations
    if aw.rule == "assign_boolean":
        if len(aw.use) != 1 or _complete(w):
            raise ValueError("Assignment rule requires exactly one next-variable branch")
        branch = aw.use[0]
        if branch not in _relevant(problem, w):
            raise ValueError("Assignment must extend the current partial certificate")
        remaining_frontier = retained - {branch}
        remaining_unexplored = unexplored - {branch}
        if remaining_frontier or remaining_unexplored:
            saved = _scout(problem, w, remaining_frontier, remaining_unexplored)
            stack = stack + (Continuation(w, saved),)
        partial = list(w.local_construction)
        partial[branch.variable] = branch.value
        next_w = replace(w, active_subgoal="solve", local_construction=tuple(partial),
                         derivation=w.derivation + (branch,), progress=w.progress + 1)
        next_u = _scout(problem, next_w)
    elif aw.rule == "restore_continuation":
        if aw.use or not stack or au.propose or aq.prune:
            raise ValueError("Restore requires an internally saved continuation")
        saved = stack[-1]
        next_w, next_u, stack = saved.worker, saved.scout, stack[:-1]
    record = s.verification_record if next_w == w else None
    next_step = s.global_step + 1
    if aq.verify:
        status = "pass" if verify_binary_assignment(problem, next_w.local_construction) else "fail"
        record = _VerifierRecord(problem.digest, _worker_digest(next_w), status, next_step)
    next_q = replace(q, budget_remaining=q.budget_remaining - 1,
                     arbitration_memory=q.arbitration_memory + (aq.decision.value,))
    next_s = TaskState(selected, record, stack, next_step,
                      aq.decision if aq.stop else AHRDecision.UNDECIDED)
    return AHRState(next_q, next_w, next_u, next_s)


def _available_ahr_actions(problem, state):
    q, w, u, s = state.queen, state.worker, state.scout, state.task
    if q.budget_remaining <= 0 or s.terminal_decision != AHRDecision.UNDECIDED:
        return ()
    if _complete(w) and verifier_status(problem, state) == "pass":
        return (AHRAction(QueenAction(decision=AHRDecision.ACCEPT, stop=True)),)
    if q.budget_remaining == 1 or _exhausted(problem, state):
        return (AHRAction(QueenAction(decision=AHRDecision.ABSTAIN, stop=True)),)
    if _complete(w):
        status = verifier_status(problem, state)
        if status == "unchecked":
            return (AHRAction(QueenAction(verify=True)),)
        if status == "pass":
            return (AHRAction(QueenAction(decision=AHRDecision.ACCEPT, stop=True)),)
        return (AHRAction(worker=WorkerAction(rule="restore_continuation")),)
    branches = tuple(sorted(u.frontier | u.unexplored))
    if not branches:
        return (AHRAction(worker=WorkerAction(rule="restore_continuation")),)
    return tuple(AHRAction(
        QueenAction(select=frozenset({"solve"})),
        WorkerAction(frozenset({"solve"}), "assign_boolean", (branch,)),
        ScoutAction(u.unexplored),
    ) for branch in branches)


def _finite_roles(scores, expected_count):
    scores = tuple(tuple(float(v) for v in row) for row in scores)
    if len(scores) != expected_count or any(len(row) != 3 or not all(math.isfinite(v) for v in row) for row in scores):
        raise ValueError("Neural scoring must return one finite (EQ, EW, EU) triple per action")
    if any(not math.isfinite(sum(row)) for row in scores):
        raise ValueError("Role energy sum overflow")
    return scores


def ahr_energy_components(problem, initial_state, trace, answer, max_expansions):
    """Eight terms on one trajectory/answer domain; UNKNOWN is not a certificate."""
    trace = tuple(trace)
    energies = {"EQ": 0.0, "EW": 0.0, "EU": 0.0,
                "Ealign": 0.0, "ET": 0.0, "EV": 0.0, "ER": 0.0, "EB": 0.0}
    cursor = initial_state
    if initial_state != initial_ahr_state(problem, max_expansions):
        energies["ET"] = math.inf
    for step in trace:
        roles = _finite_roles((step.role_energies,), 1)[0]
        for key, value in zip(("EQ", "EW", "EU"), roles):
            energies[key] += value
        if not all(qws_clauses(problem, step.before, step.action).values()):
            energies["Ealign"] = math.inf
        try:
            expected = strict_ahr_transition(problem, step.before, step.action)
            if cursor != step.before or expected != step.after:
                energies["ET"] = math.inf
        except (ValueError, IndexError, TypeError):
            energies["ET"] = math.inf
        cursor = step.after
    extraction = cursor.worker.local_construction if cursor.task.terminal_decision == AHRDecision.ACCEPT else None
    if answer is None or extraction != answer:
        energies["ER"] = math.inf
    trajectory_valid = energies["ET"] == 0 and energies["Ealign"] == 0
    if not (trajectory_valid and answer is not None and verify_binary_assignment(problem, answer)
            and cursor.task.terminal_decision == AHRDecision.ACCEPT
            and verifier_status(problem, cursor) == "pass"):
        energies["EV"] = math.inf
    if (len(trace) > max_expansions or initial_state.queen.budget_remaining != max_expansions
            or cursor.queen.budget_remaining < 0):
        energies["EB"] = math.inf
    if any(not math.isfinite(energies[key]) for key in ("EQ", "EW", "EU")):
        raise ValueError("Accumulated finite role energy overflow")
    return energies


def run_ahr_search(problem, score_fn, max_expansions=64, time_budget_s=1.0):
    """Bounded typed action-tree search. Scores rank; independent checks certify."""
    if not math.isfinite(float(time_budget_s)) or time_budget_s < 0:
        raise ValueError("time_budget_s must be finite and nonnegative")
    initial = initial_ahr_state(problem, max_expansions)
    state, trace = initial, []
    start = time.monotonic()
    deadline = start + float(time_budget_s)
    reason = "budget exhausted"
    while state.queen.budget_remaining > 0:
        if time.monotonic() >= deadline:
            reason = "deadline reached"
            break
        actions = _available_ahr_actions(problem, state)
        if not actions:
            reason = "no available action"
            break
        roles = _finite_roles(score_fn(problem, state.worker.local_construction, actions), len(actions))
        # Model callbacks cannot authorize any state/action effect. Recheck after
        # scoring so a slow callback cannot cause an out-of-deadline acceptance.
        if time.monotonic() >= deadline:
            reason = "deadline reached during scoring"
            break
        feasible = []
        for index, action in enumerate(actions):
            try:
                successor = strict_ahr_transition(problem, state, action)
            except ValueError:
                continue
            feasible.append((sum(roles[index]), index, successor))
        if not feasible:
            reason = "no feasible QWS transition"
            break
        _, index, successor = min(feasible, key=lambda row: (row[0], row[1]))
        if time.monotonic() >= deadline:
            reason = "deadline reached before transition"
            break
        trace.append(AHRTransition(state, actions[index], successor, roles[index]))
        state = successor
        if state.task.terminal_decision == AHRDecision.ACCEPT:
            reason = "independently verified certificate"
            break
        if state.task.terminal_decision == AHRDecision.ABSTAIN:
            reason = "budget abstention" if state.queen.budget_remaining == 0 else "frontier exhausted; abstention"
            break
    assignment = tuple(state.worker.local_construction) if state.task.terminal_decision == AHRDecision.ACCEPT else None
    # Include the complete final trajectory/verifier audit in the time budget.
    energy = ahr_energy_components(problem, initial, trace, assignment, max_expansions)
    elapsed = time.monotonic() - start
    accepted = assignment is not None and elapsed <= time_budget_s
    status = AHRStatus.SAT if accepted else AHRStatus.UNKNOWN
    if not accepted:
        if assignment is not None:
            reason = "deadline reached during final audit"
        assignment = None
        energy["ER"] = energy["EV"] = math.inf
    return AHRSearchResult(status, assignment, tuple(trace), len(trace), time.monotonic() - start, energy, reason, initial, state)


In [ ]:

def validate_finite_ahr_controller():
    """Synthetic structural/semantic checks, not learned strength evaluation."""
    fixture = BinaryCSP(2, ((1, 1),), "synthetic_unique")
    ascending = lambda problem, partial, actions: [(0.0, float(a.worker.use[0].value) if a.worker.use else 0.0, 0.0) for a in actions]
    descending = lambda problem, partial, actions: [(0.0, -float(a.worker.use[0].value) if a.worker.use else 0.0, 0.0) for a in actions]
    state = initial_ahr_state(fixture, 40)
    action = _available_ahr_actions(fixture, state)[0]
    assert all(qws_clauses(fixture, state, action).values())
    bad_cases = {
        "QWS-1": replace(action, queen=replace(action.queen, select=frozenset({"forbidden"}))),
        "QWS-2": replace(action, scout=ScoutAction(action.scout.propose | {Branch(9, 0)})),
        "QWS-3": replace(action, worker=replace(action.worker, use=(Branch(1, 0),))),
        "QWS-4": replace(action, queen=replace(action.queen, prune=frozenset({Branch(0, 0)}))),
        "QWS-5": AHRAction(QueenAction(decision=AHRDecision.ACCEPT)),
        "QWS-6": AHRAction(QueenAction(stop=True)),
    }
    for clause, bad in bad_cases.items():
        assert not qws_clauses(fixture, state, bad)[clause], clause
        try:
            strict_ahr_transition(fixture, state, bad)
            raise AssertionError(f"Accepted incompatible {clause}")
        except ValueError:
            pass

    result = run_ahr_search(fixture, ascending, 40, 2.0)
    assert result.status == AHRStatus.SAT and result.assignment == (1, 1)
    assert all(result.energy_breakdown[k] == 0 for k in ("Ealign", "ET", "EV", "ER", "EB"))
    assert any(t.action.worker.rule == "restore_continuation" for t in result.trace)
    assert any(t.after.task.verification_record and t.after.task.verification_record.status == "fail" for t in result.trace)
    assert result.trace[-2].action.queen.verify and result.trace[-1].action.queen.decision == AHRDecision.ACCEPT
    reversed_order = run_ahr_search(fixture, descending, 40, 2.0)
    assert reversed_order.status == AHRStatus.SAT and reversed_order.expansions < result.expansions
    exact_budget = run_ahr_search(fixture, descending, 4, 2.0)
    assert exact_budget.status == AHRStatus.SAT and exact_budget.expansions == 4

    checked = result.trace[-2].after
    assert verifier_status(fixture, checked) == "pass"
    stale = replace(checked, worker=replace(checked.worker, local_construction=(0, 0)))
    assert verifier_status(fixture, stale) == "unchecked"
    assert not qws_clauses(fixture, stale, result.trace[-1].action)["QWS-5"]
    forged_initial = replace(result.initial_state, task=replace(result.initial_state.task, global_step=99))
    initial_energy = ahr_energy_components(fixture, forged_initial, result.trace, result.assignment, 40)
    assert math.isinf(initial_energy["ET"])
    for bad_branch in ((True, 0), (0, True), (-1, 0), (0, 2)):
        try:
            Branch(*bad_branch)
            raise AssertionError("Malformed branch accepted")
        except ValueError:
            pass
    bool_worker = replace(state.worker, local_construction=(True, None))
    bool_state = replace(state, worker=bool_worker, scout=_scout(fixture, bool_worker))
    try:
        strict_ahr_transition(fixture, bool_state, AHRAction())
        raise AssertionError("Boolean silently treated as integer assignment")
    except ValueError:
        pass
    different_problem = BinaryCSP(2, ((0, 0),), "changed_constraints")
    assert verifier_status(different_problem, checked) == "unchecked"
    # Even a forged recorded 'pass' cannot override the direct membership check.
    forged = replace(stale, task=replace(stale.task, verification_record=_VerifierRecord(
        fixture.digest, _worker_digest(stale.worker), "pass", stale.task.global_step)))
    assert verifier_status(fixture, forged) == "unchecked"

    single = BinaryCSP(1, ((1,),), "synthetic_prune")
    first = initial_ahr_state(single, 10)
    prune = AHRAction(QueenAction(prune=frozenset({Branch(0, 0)})),
                      scout=ScoutAction(frozenset({Branch(0, 0), Branch(0, 1)})))
    assert all(qws_clauses(single, first, prune).values())
    after_prune = strict_ahr_transition(single, first, prune)
    assert after_prune.scout.frontier == frozenset({Branch(0, 1)})
    assert Branch(0, 0) not in after_prune.scout.unexplored
    assignment = strict_ahr_transition(single, after_prune, _available_ahr_actions(single, after_prune)[0])
    checked_single = strict_ahr_transition(single, assignment, AHRAction(QueenAction(verify=True)))
    assert verifier_status(single, checked_single) == "pass"
    assert assignment.task.verification_record is None

    wrong_answer = ahr_energy_components(fixture, result.initial_state, result.trace, (0, 0), 40)
    assert math.isinf(wrong_answer["EV"]) and math.isinf(wrong_answer["ER"])
    fabricated = replace(result.trace[-1], after=replace(result.trace[-1].after,
                        queen=replace(result.trace[-1].after.queen, budget_remaining=999)))
    malformed = ahr_energy_components(fixture, result.initial_state, result.trace[:-1] + (fabricated,), result.assignment, 40)
    assert math.isinf(malformed["ET"])
    low_invalid = -1e12 + sum(wrong_answer[k] for k in ("Ealign", "ET", "EV", "ER", "EB"))
    high_valid = 1e12 + sum(result.energy_breakdown[k] for k in ("Ealign", "ET", "EV", "ER", "EB"))
    assert math.isinf(low_invalid) and math.isfinite(high_valid) and high_valid < low_invalid
    exceeded = ahr_energy_components(fixture, result.initial_state, result.trace, result.assignment, 0)
    assert math.isinf(exceeded["EB"])

    zero = run_ahr_search(fixture, ascending, 0, 1.0)
    deadline = run_ahr_search(fixture, ascending, 40, 0.0)
    short = run_ahr_search(fixture, ascending, 1, 1.0)
    no_solution = run_ahr_search(BinaryCSP(2, (), "synthetic_no_solution"), ascending, 64, 2.0)
    assert all(r.status == AHRStatus.UNKNOWN and r.assignment is None for r in (zero, deadline, short, no_solution))
    assert zero.expansions == deadline.expansions == 0
    assert short.final_state.task.terminal_decision == AHRDecision.ABSTAIN
    assert no_solution.final_state.task.terminal_decision == AHRDecision.ABSTAIN
    assert "exhausted" in no_solution.reason
    for bad_value in (float("nan"), float("inf"), -float("inf")):
        try:
            run_ahr_search(fixture, lambda p, x, a: [(bad_value, 0, 0)] * len(a), 4, 1.0)
            raise AssertionError("Nonfinite role score accepted")
        except ValueError:
            pass
    # Deterministic clock injection isolates expiry during the FINAL audit,
    # after a verified candidate was found; no sleeps or speed assumptions.
    from unittest.mock import patch
    clock = {"now": 0.0, "audited_candidate": False}
    original_audit = ahr_energy_components
    def timed_audit(*args, **kwargs):
        clock["audited_candidate"] = args[3] is not None
        energy = original_audit(*args, **kwargs)
        clock["now"] = 2.0
        return energy
    with patch.dict(globals(), {"ahr_energy_components": timed_audit}), patch.object(time, "monotonic", lambda: clock["now"]):
        audit_expiry = run_ahr_search(fixture, descending, 40, 1.0)
    assert clock["audited_candidate"]
    assert audit_expiry.status == AHRStatus.UNKNOWN and audit_expiry.assignment is None
    assert audit_expiry.elapsed_s == 2.0 and "final audit" in audit_expiry.reason
    assert math.isinf(audit_expiry.energy_breakdown["EV"])
    return {
        "final_audit_included_in_deadline": True,
        "synthetic_only": True,
        "six_qws_clauses_reject_invalid_actions": True,
        "verifier_bound_to_problem_and_worker": True,
        "verification_precedes_acceptance": True,
        "pruning_and_selection_effects_checked": True,
        "strict_transition_tamper_rejected": True,
        "hard_energy_beats_arbitrary_soft_reward": True,
        "zero_budget_and_deadline_return_unknown": True,
        "exhaustion_remains_unknown": True,
        "learned_callback_changes_action_order": True,
        "verified_assignment": list(result.assignment),
        "ascending_order_transitions": result.expansions,
        "descending_order_transitions": reversed_order.expansions,
        "training_steps": 0,
    }


## 9. Neural energy ranking drives the finite controller

The complete truth relation for at most three Boolean variables occupies eight features. Other features encode variable count, the whole partial assignment and the proposed action. The adapter contains no reference solution selection, search call, target label or external engine. Learned Queen/Worker/Scout energy heads order the controller's actions. The pheromone prior contributes a named finite preference to the Scout score at this boundary.

This tiny AHR action-tree search demonstrates actual neural ranking followed by hard governance and independent acceptance. It is an architecture smoke, not the scalable CSP curriculum of 05 or the Go search engine of 06. The model is freshly initialized, so action order is untrained and solve counts are not evidence of learned skill.

In [ ]:
from itertools import product

def encode_binary_actions(problem, partial, candidate_actions, *, device="cpu"):
    if not candidate_actions or len(partial) != problem.n_vars:
        raise ValueError("A nonempty action set and complete partial-state schema are required")
    truth = [int(tuple(row) in problem.allowed_assignments) for row in product((0, 1), repeat=problem.n_vars)]
    truth += [0] * (8 - len(truth))
    partial_features = [float(-1 if value is None else value) for value in partial] + [-1.] * (3-problem.n_vars)
    rows = []
    for action in candidate_actions:
        if action.worker.use:
            branch = action.worker.use[0]
            variable, value, kind = branch.variable / 3, float(branch.value), 1.
        else:
            variable, value = -1., -1.
            kind = 2. if action.queen.verify else 3. if action.queen.decision == AHRDecision.ACCEPT else 4. if action.worker.rule == "restore_continuation" else 5.
        rows.append(truth + [problem.n_vars/3] + partial_features + [variable, value, sum(v is not None for v in partial)/3, kind/5])
    x = torch.tensor([rows], dtype=torch.float32, device=device)
    mask = torch.ones(x.shape[:2], dtype=torch.bool, device=device)
    return {"x": x, "valid_mask": mask, "action_mask": mask.clone()}


def make_neural_role_scorer(model):
    if model.config.input_dim != 16:
        raise ValueError("Synthetic BinaryCSP adapter defines exactly 16 features")
    if model.training:
        raise ValueError("Inference scorer requires model.eval()")
    def score(problem, partial, candidate_actions):
        reference = next(model.parameters())
        batch = encode_binary_actions(problem, partial, candidate_actions, device=reference.device)
        batch["x"] = batch["x"].to(reference.dtype)
        with torch.no_grad():
            output = model(batch)
        energies = output["energies"]
        values = torch.stack((energies["queen"], energies["worker"], energies["scout"] + energies["pheromone"]), -1)
        return values[0].float().cpu().tolist()
    return score

In [ ]:
FORMAL_CONTROLLER_VALIDATION = validate_finite_ahr_controller()
checks["formal_controller"] = FORMAL_CONTROLLER_VALIDATION
neural_score = make_neural_role_scorer(model.eval())
controller_results = []
for fixture in (BinaryCSP(2, ((1,1),), "synthetic_unique"),
                BinaryCSP(3, ((0,1,0),(1,0,1)), "synthetic_alternating"),
                BinaryCSP(2, (), "synthetic_empty")):
    result = run_ahr_search(fixture, neural_score, max_expansions=64, time_budget_s=5.0)
    expected = AHRStatus.SAT if fixture.allowed_assignments else AHRStatus.UNKNOWN
    assert result.status == expected
    if result.status == AHRStatus.SAT:
        assert verify_binary_assignment(fixture, result.assignment)
        assert all(result.energy_breakdown[key] == 0 for key in ("Ealign", "ET", "EV", "ER", "EB"))
    controller_results.append({"fixture": fixture.name, "status": result.status.value,
        "assignment": result.assignment, "transitions": result.expansions,
        "elapsed_seconds": round(result.elapsed_s, 4), "reason": result.reason,
        "finite_soft_energy": sum(result.energy_breakdown[key] for key in ("EQ", "EW", "EU")),
        "hard_energy_zero": all(result.energy_breakdown[key] == 0 for key in ("Ealign", "ET", "EV", "ER", "EB"))})
assert all(torch.equal(value, INITIAL_WEIGHTS[name]) for name, value in model.state_dict().items())
checks["neural_controller_integration"] = controller_results
print("All six QWS rejection tests, stale verification and hard feasibility: PASS")
for result in controller_results:
    print(result["fixture"], result["status"], result["assignment"], result["transitions"], "typed transitions")

## 10. Export interruption-safe recovery for nested state

Every validation stage is checkpointed. The manifest retains the latest two plus the untrained reference initialization. For this notebook only, `global_step` is a validation-stage counter; **optimizer_steps is always zero**. “Best” means the reference initialization, not the strongest model. Optimizer/scheduler/scaler/adapters are null, sampler is empty, replay is null and distributed world size is one.

The checkpoint contains model weights/mode, Q/W/U, pheromone, both partial summaries, counters, input/config identity, fixture, metrics and all RNGs. It rejects the old prototype schema. A run has one writer; malformed or corrupted artifacts fail explicitly. This short test validates architecture recovery only; future long training needs its own optimizer/sampler/distributed resume test.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import copy, hashlib, json, random
import numpy as np
import torch

ARCHITECTURE_RESUME_CELL = "04: Architecture checkpoint validation — Run All (AHR04_PHASE=resume)"

def architecture_cpu_tree(value):
    """Detach a checkpoint from graphs/devices without retaining live tensors."""
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: architecture_cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [architecture_cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(architecture_cpu_tree(item) for item in value)
    return copy.deepcopy(value)

def architecture_equal(left, right):
    if isinstance(left, torch.Tensor):
        return isinstance(right, torch.Tensor) and left.shape == right.shape and left.dtype == right.dtype and torch.equal(left.cpu(), right.cpu())
    if isinstance(left, np.ndarray):
        return isinstance(right, np.ndarray) and np.array_equal(left, right)
    if isinstance(left, dict):
        return isinstance(right, dict) and left.keys() == right.keys() and all(architecture_equal(left[key], right[key]) for key in left)
    if isinstance(left, (tuple, list)):
        return type(left) is type(right) and len(left) == len(right) and all(architecture_equal(a, b) for a, b in zip(left, right))
    return left == right

def architecture_parameter_digest(model):
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str((value.dtype, tuple(value.shape))).encode())
        digest.update(value.reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()

def architecture_rng_probe():
    return {"python": random.random(), "numpy": np.random.random(4), "torch_cpu": torch.rand(4),
            "torch_cuda": [torch.rand(4, device=f"cuda:{index}").cpu() for index in range(torch.cuda.device_count())]}

def architecture_state(model, batch, stage, metrics, recurrent=None, *, reference_digest=None):
    """global_step counts validation stages ONLY; optimizer_steps remains zero."""
    if type(stage) is not int or stage not in (1, 2, 3):
        raise ValueError("Architecture validation has exactly three checkpoint stages")
    return architecture_cpu_tree({
        "schema": "ahr04-architecture-v2", "model": model.state_dict(), "adapters": None,
        "optimizer": None, "scheduler": None, "scaler": None, "optimizer_steps": 0,
        "no_training": True, "model_training_mode": model.training,
        "rng": capture_rng(), "sampler": {"dataset": None, "position": 0, "epoch": 0},
        "replay": None, "distributed": {"world_size": 1, "rank": 0},
        "global_step": stage, "global_step_semantics": "architecture_validation_stage",
        "config": asdict(model.config), "metrics": metrics, "batch": batch,
        "recurrent_state": recurrent, "reference_parameter_digest": reference_digest or architecture_parameter_digest(model),
        "best_semantics": "reference untrained initialization; no learned quality comparison",
    })

def architecture_validate_payload(state):
    required = REQUIRED_CHECKPOINT_KEYS | {"schema", "adapters", "optimizer_steps", "no_training", "model_training_mode", "global_step_semantics", "batch", "recurrent_state", "reference_parameter_digest", "replay", "distributed", "best_semantics"}
    if not isinstance(state, dict) or required - set(state):
        raise RuntimeError("Architecture checkpoint has missing required fields")
    if state["schema"] != "ahr04-architecture-v2" or type(state["global_step"]) is not int or state["global_step"] not in (1, 2, 3):
        raise RuntimeError("Unsupported architecture checkpoint schema or stage")
    if state["global_step_semantics"] != "architecture_validation_stage" or state["optimizer_steps"] != 0 or state["no_training"] is not True:
        raise RuntimeError("Architecture checkpoint violates the no-training contract")
    if any(state[key] is not None for key in ("optimizer", "scheduler", "scaler", "adapters", "replay")):
        raise RuntimeError("Unexpected training components in architecture checkpoint")
    if state["sampler"] != {"dataset": None, "position": 0, "epoch": 0} or state["distributed"] != {"world_size": 1, "rank": 0}:
        raise RuntimeError("Unexpected dataset or distributed state in architecture checkpoint")
    if type(state["model_training_mode"]) is not bool or not isinstance(state["metrics"], dict):
        raise RuntimeError("Invalid model mode or checkpoint metrics")
    try:
        checkpoint_config = AHRConfig(**state["config"])
    except (TypeError, ValueError) as error:
        raise RuntimeError("Invalid architecture configuration") from error
    recurrent = state["recurrent_state"]
    if state["global_step"] == 1 and recurrent is not None:
        raise RuntimeError("Initialization checkpoint must precede recurrent execution")
    if state["global_step"] >= 2:
        required_recurrent = {"queen", "worker", "scout", "pheromone", "scout_summary", "worker_summary",
                              "iteration", "worker_index", "scout_index", "micro_step", "input_digest", "config_digest"}
        if not isinstance(recurrent, dict) or set(recurrent) != required_recurrent:
            raise RuntimeError("Checkpoint lacks the complete resumable nested AHR state")
        for key in ("iteration", "worker_index", "scout_index", "micro_step"):
            if type(recurrent[key]) is not int or recurrent[key] < 0:
                raise RuntimeError("Invalid nested recurrence cursor")
        cycle_size = checkpoint_config.worker_steps * checkpoint_config.scout_steps
        if (recurrent["iteration"] > checkpoint_config.max_iterations
                or recurrent["worker_index"] >= checkpoint_config.worker_steps
                or recurrent["scout_index"] >= checkpoint_config.scout_steps
                or recurrent["micro_step"] != recurrent["iteration"] * cycle_size
                    + recurrent["worker_index"] * checkpoint_config.scout_steps + recurrent["scout_index"]):
            raise RuntimeError("Inconsistent nested recurrence cursor")
        if recurrent["micro_step"] > checkpoint_config.max_iterations * cycle_size:
            raise RuntimeError("Checkpoint exceeds the bounded recurrence budget")
        fixture = state["batch"]
        if not isinstance(fixture, dict) or set(fixture) != {"x", "valid_mask", "action_mask"}:
            raise RuntimeError("Checkpoint input contract is incomplete")
        b, length, _ = fixture["x"].shape
        for key in ("queen", "worker", "scout", "scout_summary", "worker_summary"):
            if not isinstance(recurrent[key], torch.Tensor) or recurrent[key].shape != (b, checkpoint_config.hidden_dim) or not torch.isfinite(recurrent[key]).all():
                raise RuntimeError("Invalid checkpoint role state or partial summary")
        if not isinstance(recurrent["pheromone"], torch.Tensor) or recurrent["pheromone"].shape != (b, length) or not torch.isfinite(recurrent["pheromone"]).all():
            raise RuntimeError("Invalid pheromone checkpoint state")
        for key in ("input_digest", "config_digest"):
            if not isinstance(recurrent[key], str) or len(recurrent[key]) != 64:
                raise RuntimeError("Checkpoint lacks its bound input/config identity")
        for metric, key in (("recurrent_iteration", "iteration"), ("recurrent_micro_step", "micro_step")):
            if state["metrics"].get(metric) != recurrent[key]:
                raise RuntimeError("Checkpoint recurrence metrics do not match its cursor")
    if state["global_step"] >= 2 and "expected_continuation" not in state:
        raise RuntimeError("Checkpoint lacks its exact-resume reference")
    if state["global_step"] == 3 and state["metrics"].get("exact_resume") is not True:
        raise RuntimeError("Completed checkpoint lacks successful resume verification")
    return state

def architecture_load_unlocked(run_dir):
    """Fail closed on malformed manifests or any corrupt committed payload."""
    directory = Path(run_dir)
    path = directory / "manifest.json"
    if not path.exists():
        if list(directory.glob("*.pt")):
            raise RuntimeError("Architecture payloads exist without a manifest; inspect this run instead of restarting")
        return None
    try:
        manifest = json.loads(path.read_text())
        entries = manifest["entries"]
        if manifest["schema_version"] != 1 or not isinstance(entries, list) or not entries:
            raise ValueError("Manifest schema or entries invalid")
        files = [entry["file"] for entry in entries]
        steps = [entry["global_step"] for entry in entries]
        if len(set(files)) != len(files) or len(set(steps)) != len(steps):
            raise ValueError("Duplicate manifest entries")
        if any(type(step) is not int or step not in (1, 2, 3) for step in steps):
            raise ValueError("Invalid architecture validation stage")
        latest = max(entries, key=lambda entry: entry["global_step"])
        if manifest["latest"] != latest["file"] or manifest["best"] not in files or manifest["resume_cell"] != ARCHITECTURE_RESUME_CELL:
            raise ValueError("Manifest pointers or resume instructions invalid")
        states = {}
        for entry in entries:
            payload = _checkpoint_entry_state(directory, entry)
            if payload is None:
                raise ValueError(f"Missing, corrupt, or hash-mismatched committed payload: {entry['file']}")
            states[entry["file"]] = architecture_validate_payload(payload)
        if states[manifest["best"]]["global_step"] != 1:
            raise ValueError("Best pointer must retain the untrained initialization")
        if max(steps) >= 2 and not {max(steps) - 1, max(steps)}.issubset(steps):
            raise ValueError("Manifest does not retain the latest two stages")
        return states[manifest["latest"]]
    except Exception as error:
        raise RuntimeError(f"Invalid architecture checkpoint or manifest at {directory}; refusing silent restart") from error

def restore_architecture_checkpoint(state, *, restore_rng_state=True):
    """CPU portable model restore. Exact RNG restore also requires matching CUDA topology."""
    architecture_validate_payload(state)
    model = NativeAHR(AHRConfig(**state["config"])).cpu()
    model.load_state_dict(state["model"], strict=True)
    model.train(state["model_training_mode"])
    if architecture_parameter_digest(model) != state["reference_parameter_digest"]:
        raise RuntimeError("Checkpoint parameters differ from the untrained initialization")
    fixture = architecture_cpu_tree(state["batch"])
    recurrent = architecture_cpu_tree(state["recurrent_state"])
    model._validate_batch(fixture)
    if recurrent is not None:
        model._validate_state(recurrent, fixture, _ahr_batch_digest(fixture))
    if restore_rng_state:
        if len(state["rng"]["torch_cuda"]) != torch.cuda.device_count():
            raise RuntimeError("Exact RNG resume requires the checkpoint's CUDA device count; use restore_rng_state=False only for portable inference")
        restore_rng(state["rng"])
    return model, fixture, recurrent

def architecture_checkpoint_smoke(project, config, batch, *, phase="all", run_id="architecture04-smoke-v2", seed=20260923):
    """Forward-only deterministic replay; holds a single writer lock for the entire lifecycle."""
    if phase not in ("all", "interrupt", "resume"):
        raise ValueError("phase must be all, interrupt, or resume")
    if not isinstance(run_id, str) or not run_id or Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("Use a simple nonempty run ID")
    if config.dropout <= 0:
        raise ValueError("Exact-resume smoke must exercise stochastic dropout")
    if config.scout_steps < 2 or config.worker_steps < 2 or config.max_iterations < 2:
        raise ValueError("Replay requires an interior Scout checkpoint and the following Queen cycle")
    continuation_micro_steps = config.worker_steps * config.scout_steps + 1
    directory = Path(project) / "checkpoints" / run_id
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    with checkpoint_lock(directory):
        existing = architecture_load_unlocked(directory)
        resumed_from = None if existing is None else existing["global_step"]
        if existing is None and phase == "resume":
            raise RuntimeError("Run the interrupt phase first; resume never creates a new run")
        if existing is not None:
            if existing["config"] != asdict(config) or not architecture_equal(existing["batch"], architecture_cpu_tree(batch)):
                raise RuntimeError("Config or fixture changed; use the original inputs or a new run ID")
        else:
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
            model = NativeAHR(config).cpu()
            # train() activates dropout; no loss, backward, optimizer, or parameter update occurs.
            model.train()
            existing = architecture_state(model, batch, 1, {"stage": "initialization", "optimizer_steps": 0})
            _save_checkpoint_unlocked(directory, existing, best=True, resume_cell=ARCHITECTURE_RESUME_CELL)
            existing = architecture_load_unlocked(directory)
        if existing["global_step"] == 1:
            model, fixture, _ = restore_architecture_checkpoint(existing)
            with torch.no_grad():
                partial = model(fixture, micro_steps=1)
            if partial["state"]["iteration"] != 0 or partial["state"]["worker_index"] != 0 or partial["state"]["scout_index"] != 1 or partial["state"]["micro_step"] != 1:
                raise AssertionError("Smoke checkpoint must stop inside the first Scout loop")
            interrupted = architecture_state(model, fixture, 2, {"stage": "interrupted", "optimizer_steps": 0, "recurrent_iteration": partial["state"]["iteration"], "recurrent_micro_step": partial["state"]["micro_step"]}, partial["state"], reference_digest=existing["reference_parameter_digest"])
            with torch.no_grad():
                uninterrupted = model(fixture, micro_steps=continuation_micro_steps, state=partial["state"])
            interrupted["expected_continuation"] = architecture_cpu_tree({"output": uninterrupted, "rng_probe": architecture_rng_probe(), "rng_after": capture_rng()})
            if architecture_parameter_digest(model) != interrupted["reference_parameter_digest"]:
                raise AssertionError("Forward-only validation unexpectedly changed model state")
            _save_checkpoint_unlocked(directory, interrupted, resume_cell=ARCHITECTURE_RESUME_CELL)
            existing = architecture_load_unlocked(directory)
        if phase != "interrupt" and existing["global_step"] == 2:
            model, fixture, recurrent = restore_architecture_checkpoint(existing)
            with torch.no_grad():
                continued = model(fixture, micro_steps=continuation_micro_steps, state=recurrent)
            observed = architecture_cpu_tree({"output": continued, "rng_probe": architecture_rng_probe(), "rng_after": capture_rng()})
            if not architecture_equal(observed, existing["expected_continuation"]):
                raise AssertionError("Stochastic recurrent continuation or RNG differs from uninterrupted execution")
            final = architecture_state(model, fixture, 3, {"stage": "complete", "optimizer_steps": 0, "exact_resume": True, "recurrent_iteration": continued["state"]["iteration"], "recurrent_micro_step": continued["state"]["micro_step"]}, continued["state"], reference_digest=existing["reference_parameter_digest"])
            final["expected_continuation"] = existing["expected_continuation"]
            _save_checkpoint_unlocked(directory, final, resume_cell=ARCHITECTURE_RESUME_CELL)
            existing = architecture_load_unlocked(directory)
        model, _, recurrent = restore_architecture_checkpoint(existing)
        if existing["global_step"] == 3 and not architecture_equal(recurrent, existing["expected_continuation"]["output"]["state"]):
            raise AssertionError("Completed checkpoint's recurrent state does not match its verified continuation")
        manifest = json.loads((directory / "manifest.json").read_text())
        return {"run_id": run_id, "resumed_from_stage": resumed_from, "validation_stage": existing["global_step"],
                "optimizer_steps": 0, "exact_resume": existing["metrics"].get("exact_resume", False),
                "reference_parameter_digest": architecture_parameter_digest(model), "metrics": existing["metrics"],
                "retained_stages": sorted(entry["global_step"] for entry in manifest["entries"]),
                "checkpoint_micro_step": 1, "continuation_micro_steps": continuation_micro_steps,
                "resume_cell": manifest["resume_cell"], "manifest": str(directory / "manifest.json")}

def architecture_pin_provenance(run_dir, provenance):
    """Pin source and software identity before opening this authored run's payloads."""
    directory = Path(run_dir)
    with checkpoint_lock(directory):
        identity_path = directory / "source.json"
        if identity_path.exists():
            try:
                saved = json.loads(identity_path.read_text())
            except (OSError, ValueError) as error:
                raise RuntimeError("Checkpoint provenance is invalid; refusing silent restart") from error
            if saved != provenance:
                raise RuntimeError("Code or environment differs from this run; restore the original or choose a new run ID")
        elif (directory / "manifest.json").exists() or list(directory.glob("*.pt")):
            raise RuntimeError("Checkpoint provenance is missing; refusing silent restart")
        else:
            atomic_json(identity_path, provenance)

def architecture_checkpoint_fault_checks(project, config, batch, *, run_id, provenance):
    """Exercise recovery faults on disposable copies, retaining the committed run."""
    import tempfile, shutil
    directory = Path(project) / "checkpoints" / run_id
    rejected = []
    def expect_failure(name, operation):
        try:
            operation()
        except (RuntimeError, ValueError, OSError):
            rejected.append(name)
        else:
            raise AssertionError(f"Checkpoint fault not rejected: {name}")
    with tempfile.TemporaryDirectory(prefix="fault-check-", dir=directory) as temporary:
        fault_root = Path(temporary)
        def copy_snapshot(name):
            target = fault_root / name
            target.mkdir()
            for file in directory.iterdir():
                if file.is_file() and (file.name in {"manifest.json", "source.json"} or file.suffix == ".pt"):
                    shutil.copy2(file, target / file.name)
            return target
        malformed = copy_snapshot("malformed")
        (malformed / "manifest.json").write_text("{invalid json")
        expect_failure("malformed_manifest", lambda: architecture_load_unlocked(malformed))
        corrupt = copy_snapshot("corrupt")
        manifest = json.loads((corrupt / "manifest.json").read_text())
        (corrupt / manifest["latest"]).write_bytes(b"invalid checkpoint")
        expect_failure("hash_mismatch", lambda: architecture_load_unlocked(corrupt))
        missing = copy_snapshot("missing")
        (missing / "manifest.json").unlink()
        expect_failure("missing_manifest", lambda: architecture_load_unlocked(missing))
        with checkpoint_lock(fault_root / "writer"):
            try:
                with checkpoint_lock(fault_root / "writer"):
                    pass
            except RuntimeError as error:
                assert "Another process owns this run" in str(error)
                rejected.append("second_writer")
            else:
                raise AssertionError("A second writer acquired the same run lock")
        pointer = copy_snapshot("pointer")
        manifest = json.loads((pointer / "manifest.json").read_text())
        manifest["latest"] = "missing.pt"
        (pointer / "manifest.json").write_text(json.dumps(manifest))
        expect_failure("invalid_latest_pointer", lambda: architecture_load_unlocked(pointer))
        missing_source = copy_snapshot("missing-source")
        (missing_source / "source.json").unlink()
        expect_failure("missing_provenance", lambda: architecture_pin_provenance(missing_source, provenance))
        changed_source = copy_snapshot("changed-source")
        expect_failure("changed_provenance", lambda: architecture_pin_provenance(changed_source, {**provenance, "source_sha256": "0" * 64}))
    altered_config = AHRConfig(**{**asdict(config), "hidden_dim": config.hidden_dim + 8})
    expect_failure("changed_config", lambda: architecture_checkpoint_smoke(project, altered_config, batch, run_id=run_id))
    changed_batch = {**batch, "x": batch["x"] + 0.01}
    expect_failure("changed_fixture", lambda: architecture_checkpoint_smoke(project, config, changed_batch, run_id=run_id))
    invalid_cursor = copy.deepcopy(architecture_load_unlocked(directory))
    invalid_cursor["recurrent_state"]["micro_step"] += 1
    expect_failure("inconsistent_recurrence_cursor", lambda: architecture_validate_payload(invalid_cursor))
    missing_summary = copy.deepcopy(architecture_load_unlocked(directory))
    missing_summary["recurrent_state"].pop("scout_summary")
    expect_failure("missing_partial_summary", lambda: architecture_validate_payload(missing_summary))
    prototype = copy.deepcopy(architecture_load_unlocked(directory))
    prototype["schema"] = "ahr04-architecture-v1"
    expect_failure("prototype_schema", lambda: architecture_validate_payload(prototype))
    for key in ("input_digest", "config_digest"):
        wrong_identity = copy.deepcopy(architecture_load_unlocked(directory))
        wrong_identity["recurrent_state"][key] = "0" * 64
        expect_failure("restore_" + key, lambda state=wrong_identity: restore_architecture_checkpoint(state, restore_rng_state=False))
    wrong_bound = copy.deepcopy(architecture_load_unlocked(directory))
    wrong_bound["recurrent_state"]["pheromone"] += 100.0
    expect_failure("restore_pheromone_bound", lambda: restore_architecture_checkpoint(wrong_bound, restore_rng_state=False))
    wrong_dtype = copy.deepcopy(architecture_load_unlocked(directory))
    wrong_dtype["recurrent_state"]["scout"] = wrong_dtype["recurrent_state"]["scout"].double()
    expect_failure("restore_state_dtype", lambda: restore_architecture_checkpoint(wrong_dtype, restore_rng_state=False))
    return rejected


## 11. Architecture checkpoint validation — Run All and exact resume

**Restart Kernel and Run All** executes all checks and creates/reuses a source-pinned run under `checkpoints/architecture-04-theory-<source-hash>/`. The snapshot is taken after **Scout microstep 1**. Stochastic dropout continuation crosses both Worker and Queen boundaries and must reproduce all outputs, traces and RNGs exactly on the same software/device topology.

For a fresh-kernel interruption test, run these commands in order from the workspace, keeping source/config/fixture unchanged:

```text
AHR04_PHASE=interrupt AHR04_RUN_ID=architecture-04-theory-fresh-restart .venv/bin/python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=ahr-go notebooks/04_native_ahr_architecture.ipynb
AHR04_PHASE=resume AHR04_RUN_ID=architecture-04-theory-fresh-restart .venv/bin/python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=ahr-go notebooks/04_native_ahr_architecture.ipynb
```

The first run reports `INTERRUPTED_FOR_RESUME_TEST`, stage 2. The second must report `resumed_from_stage=2`, stage 3, `exact_resume=true`, retained stages `[1,2,3]`, and zero optimizer steps. With defaults the resumed state ends at microstep 8. Completed reruns verify saved state and metrics without overwriting.

Schema/config/source mismatches require the matching original code or an explicit new run ID; invalid existing runs never silently restart. Model-only CPU loading can disable RNG restoration, while exact continuation requires the saved topology. Full execution from the workspace without the UI uses the same command with `AHR04_PHASE` omitted.

In [ ]:
own_notebook = nbformat.read(NOTEBOOK_PATH, as_version=4)
SOURCE_SHA256 = hashlib.sha256("\n".join(cell.source for cell in own_notebook.cells if cell.cell_type == "code").encode()).hexdigest()
EXPORT_SHA256 = hashlib.sha256("\n".join(cell.source for cell in own_notebook.cells if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", [])).encode()).hexdigest()
PHASE = os.environ.get("AHR04_PHASE", "all")
RUN_ID = os.environ.get("AHR04_RUN_ID", "architecture-04-theory-" + SOURCE_SHA256[:12])
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in (".", ".."):
    raise ValueError("AHR04_RUN_ID must be a simple nonempty name inside checkpoints")
RUN_DIR = PROJECT / "checkpoints" / RUN_ID
provenance = {"schema": "ahr04-source-v2", "implementation": AHR_IMPLEMENTATION,
              "source_sha256": SOURCE_SHA256, "recovery_export_sha256": RECOVERY_SOURCE_SHA256,
              "theory_sha256": THEORY_SHA256, "torch": torch.__version__, "python": platform.python_version(),
              "cuda_version": torch.version.cuda,
              "cuda_devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
architecture_pin_provenance(RUN_DIR, provenance)
CHECKPOINT_CHECK = architecture_checkpoint_smoke(PROJECT, CONFIG, BATCH, phase=PHASE, run_id=RUN_ID, seed=SEED)
checks["checkpoint"] = CHECKPOINT_CHECK
checks["checkpoint_fault_rejections"] = architecture_checkpoint_fault_checks(PROJECT, CONFIG, BATCH, run_id=RUN_ID, provenance=provenance)
print(json.dumps({key: CHECKPOINT_CHECK[key] for key in ("validation_stage", "resumed_from_stage", "exact_resume", "checkpoint_micro_step", "continuation_micro_steps", "optimizer_steps", "retained_stages")}, indent=2))
print("Recovery faults rejected:", len(checks["checkpoint_fault_rejections"]))

## 12. Definition-only import and delivery receipt

The model and finite controller import from 04 alone by executing `ahr-export` cells. Recovery helpers additionally require notebook 00's tagged definitions. Upstream training/generation cells are never executed. The saved report distinguishes the continuous neural realization, exact finite fixture and untested claims.

In [ ]:
import sys, types
export_module = types.ModuleType("ahr04_theory_import_validation")
sys.modules[export_module.__name__] = export_module
try:
    exports = [cell for cell in own_notebook.cells if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", [])]
    for index, cell in enumerate(exports):
        exec(compile(cell.source, f"04:ahr-export-{index}", "exec"), export_module.__dict__)
    imported = export_module.NativeAHR(export_module.AHRConfig()).eval()
    imported.load_state_dict(INITIAL_WEIGHTS)
    with torch.no_grad():
        close(imported(BATCH)["policy_logits"], baseline["policy_logits"], atol=0, rtol=0)
    assert not hasattr(export_module, "FORMAL_CONTROLLER_VALIDATION"), "A workflow ran during definition import"
finally:
    sys.modules.pop(export_module.__name__, None)
checks["definition_only_import"] = {"passed": True, "export_cells": len(exports), "upstream_workflows_executed": 0}
assert all(torch.equal(value, INITIAL_WEIGHTS[name]) for name, value in model.state_dict().items())
mount = subprocess.run(["findmnt", "-J", "-T", str(PROJECT), "-o", "SOURCE,TARGET,FSTYPE"], capture_output=True, text=True, check=True)
mount_info = json.loads(mount.stdout)["filesystems"][0]
source_device = Path(mount_info["source"]).resolve()
disk_ids = sorted(p.name for p in Path("/dev/disk/by-id").glob("*") if p.resolve() == source_device)
REPORT = {"schema_version": 2, "status": "PASS" if CHECKPOINT_CHECK["validation_stage"] == 3 else "INTERRUPTED_FOR_RESUME_TEST",
          "checked_utc": datetime.now(timezone.utc).isoformat(), "notebook": str(NOTEBOOK_PATH.relative_to(PROJECT)),
          "implementation": AHR_IMPLEMENTATION, "theory": {"file": THEORY_PATH.name, "sha256": THEORY_SHA256,
          "mapped_sections": ["4.1-4.4", "5", "6", "7", "8.1-8.2", "15", "23"]},
          "source_sha256": SOURCE_SHA256, "export_sha256": EXPORT_SHA256,
          "recovery_export_sha256": RECOVERY_SOURCE_SHA256, "config": asdict(CONFIG), "parameter_count": PARAMETER_COUNT,
          "optimizer_steps": 0, "datasets_opened": [], "checks": checks,
          "environment": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
             "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]},
          "storage": {"mount": mount_info, "matching_disk_ids": disk_ids,
             "persistent_boot_disk_identified": any("google-" in name or "Google_PersistentDisk" in name for name in disk_ids),
             "instance_delete_policy": "UNVERIFIED", "off_instance_backup_created": False},
          "elapsed_seconds": round(time.monotonic() - started_at, 3),
          "limits": ["Untrained continuous neural realization with a bounded finite integration fixture",
             "No full-game Go player, optimality, transfer or learned-strength claim", "No global energy-minimizer claim",
             "No learned coverage, admissible pruning or calibrated halting guarantee",
             "Cooperative operation-boundary deadlines, not hard real-time preemption",
             "No distributed training validation; future training needs its own resume smoke"]}
atomic_json(PROJECT / "config/04_architecture.json", {"implementation": AHR_IMPLEMENTATION, "config": asdict(CONFIG),
    "source_sha256": SOURCE_SHA256, "theory_sha256": THEORY_SHA256, "training_authorized": False})
atomic_json(PROJECT / "reports/04_architecture_validation.json", REPORT)
atomic_json(RUN_DIR / ("validation-" + PHASE + ".json"), REPORT)
from IPython.display import Markdown, display
display(Markdown(f"**{REPORT['status']}** · {PARAMETER_COUNT:,} untrained parameters · "
    f"{len(DEVICE_CHECKS)} devices checked · **0 optimizer updates** · {REPORT['elapsed_seconds']:.1f} seconds.\n\n"
    f"Nested Q/W/U, pheromone, hard QWS/verification and recovery checks are recorded in `reports/04_architecture_validation.json`. "
    f"Checkpoints: `{RUN_DIR.relative_to(PROJECT)}`."))

## Handoff and interpretation

The exported core is `AHRConfig`, `NativeAHR`, `Queen`, `Worker`, `Scout`, `StigmergicMemory`, `pheromone_update`, `pheromone_policy`, and `detach_ahr_state`. The typed finite boundary exports `BinaryCSP`, `AHRState`, `AHRAction`, `qws_clauses`, `strict_ahr_transition`, `ahr_energy_components`, `verify_binary_assignment`, and `run_ahr_search`. The neural adapter is explicitly limited to the synthetic Boolean truth-table representation.

This rewrite maps the supplied theory to executable mechanisms and records its implementation choices. The tiny controller validates role governance and verified emission; it does not certify a general Go strategy. Notebook 05 still needs the training representation, objectives and independent resume validation. Notebook 06 still needs complete Go/history inputs and timed AHR move selection. Neither is executed here.

The workspace is on persistent boot storage; instance-deletion policy remains unverified. Keep a hash-verified off-instance copy of required artifacts before recycling/deleting the instance. This notebook makes no off-instance backup. The superseded prototype remains archived under `reports/archive/` for provenance.